In [1]:
suppressPackageStartupMessages({
  library(dplyr)
  library(readr)
  library(stringr)
  library(purrr)
  library(tidyr)
  library(ggplot2)
  library(MESS)
  library(drc)
  library(ggrepel)
  library(colorspace)
  library(patchwork)
  library(cowplot)
})

select <- dplyr::select
filter <- dplyr::filter

Warning message:
“package ‘readr’ was built under R version 4.2.3”
Warning message:
“package ‘stringr’ was built under R version 4.2.3”
Warning message:
“package ‘purrr’ was built under R version 4.2.3”
Warning message:
“package ‘tidyr’ was built under R version 4.2.3”
Warning message:
“package ‘ggplot2’ was built under R version 4.2.3”
Warning message:
“package ‘MESS’ was built under R version 4.2.3”
Warning message:
“package ‘drc’ was built under R version 4.2.3”
Warning message:
“package ‘MASS’ was built under R version 4.2.3”
Warning message:
“package ‘ggrepel’ was built under R version 4.2.3”
Warning message:
“package ‘colorspace’ was built under R version 4.2.3”
Warning message:
“package ‘patchwork’ was built under R version 4.2.3”
Warning message:
“package ‘cowplot’ was built under R version 4.2.3”


In [2]:
cell_killing_file <- file.path("./data/cell-killing.csv")
raw <- read_csv(cell_killing_file, col_names = FALSE, show_col_types = FALSE)
colnames(raw) <- c("V1","V2","V3","V4","V5")
 
results <- list()
current_sample <- NA_character_
current_drug   <- NA_character_
 
for (i in seq_len(nrow(raw))) {
 
  if (!is.na(raw$V1[i])) {
    current_sample <- raw$V1[i]
    current_drug   <- raw$V2[i]
    next
  }
 
  if (!is.na(raw$V2[i]) && grepl("nM", raw$V2[i], ignore.case = TRUE)) {
    current_drug <- raw$V2[i]
    next
  }
 
  conc <- suppressWarnings(as.numeric(raw$V2[i]))
 
  if (!is.na(conc)) {
    results[[length(results) + 1]] <- tibble(
      sample = current_sample,
      drug = current_drug,
      concentration_nM = conc,
      surviving_fraction = mean(
        c(
          suppressWarnings(as.numeric(raw$V3[i])),
          suppressWarnings(as.numeric(raw$V4[i])),
          suppressWarnings(as.numeric(raw$V5[i]))
        ),
        na.rm = TRUE
      )
    )
  }
}
 
cell_killing_long <- bind_rows(results)
 
# Sanity check: every (sample, drug) should have ~10 dose points
stopifnot(nrow(cell_killing_long) > 0)
message("Parsed ", nrow(cell_killing_long), " concentration/SF rows across ",
        n_distinct(cell_killing_long$sample), " samples and ",
        n_distinct(cell_killing_long$drug), " raw drug labels.")
print(cell_killing_long %>% count(sample, drug))

Parsed 268 concentration/SF rows across 9 samples and 6 raw drug labels.



# A tibble: 27 × 3
   sample      drug                         n
   <chr>       <chr>                    <int>
 1 BT245       3-Deazaneplanocin A (nM)    10
 2 BT245       Axitinib ( nM)              10
 3 BT245       Cladribine (nM)             10
 4 DIPG10 (WT) 3-Deazaneplanocin A (nM)    10
 5 DIPG10 (WT) Axitinib (nM)               10
 6 DIPG10 (WT) Cladribine (nM)             10
 7 DIPG13 (M)  3-Deazaneplanocin A (nM)    10
 8 DIPG13 (M)  Axitinib (nM)               10
 9 DIPG13 (M)  Cladribine (nM)             10
10 DIPG17 (M)  3-Deazaneplanocin A (nM)    10
# ℹ 17 more rows


In [3]:
final_drug_file <- file.path("./results/final_drug_w_paths.csv")
final_drug_w_paths <- read_csv(final_drug_file)
final_drug_w_paths <- final_drug_w_paths %>%
  mutate(ModelID = if_else(ModelID == "GSM7305243", "MAF-868", ModelID))


New names:
• `` -> `...1`
Rows: 90 Columns: 15
── Column specification ────────────────────────────────────────────────────────
Delimiter: ","
chr (3): ModelID, model, name
dbl (7): ...1, latent_score, z, latent_dim_total, init, pearson_correlation,...
lgl (5): moa, target, indication, phase, Associated Pathways

ℹ Use `spec()` to retrieve the full column specification for this data.
ℹ Specify the column types or set `show_col_types = FALSE` to quiet this message.


In [4]:
auc_df <- cell_killing_long %>%
  arrange(sample, drug, concentration_nM) %>%
  group_by(sample, drug) %>%
  mutate(
    x = log10(concentration_nM + 1),
    x_scaled = (x - min(x)) / (max(x) - min(x))
  ) %>%
  summarize(
    auc = MESS::auc(x = x_scaled, y = surviving_fraction),
    .groups = "drop"
  )


In [5]:
cell_killing_df <- auc_df

In [6]:
clean_sample <- function(x) {
  x <- str_remove(x, "\\s*\\(.*\\)")     # remove "(WT)", "(M)"
  x <- str_remove(x, "_SHC\\d+")          # remove "_SHC202" etc.
  x <- str_trim(x)
  x <- str_replace(x, "^MAF868$", "MAF-868")  # explicit fix for the hyphen
  x
}
 
clean_drug <- function(x) {
  x_lower <- str_to_lower(x)
  case_when(
    str_detect(x_lower, "deazaneplanocin") ~ "3-deazaneplanocin-a",
    str_detect(x_lower, "axitinib")        ~ "axitinib",
    str_detect(x_lower, "cladribine")      ~ "cladribine",
    TRUE ~ str_squish(str_remove(x_lower, "\\s*\\(.*\\)"))
  )
}
 
cell_killing_df <- cell_killing_df %>%
  mutate(
    sample = clean_sample(sample),
    drug   = clean_drug(drug)
  ) %>%
  rename(ModelID = sample, name = drug)
 
final_drug_w_paths <- final_drug_w_paths %>%
  mutate(
    ModelID = clean_sample(ModelID),
    name    = clean_drug(name)
  )


In [ ]:
message("\nDistinct (ModelID, name) in cell_killing_df not found in final_drug_w_paths:")
unmatched <- anti_join(cell_killing_df, final_drug_w_paths, by = c("ModelID", "name")) %>%
  filter(name %in% c("3-deazaneplanocin-a", "cladribine", "axitinib")) %>%
  select(ModelID, name)
print(unmatched)



Distinct (ModelID, name) in cell_killing_df not found in final_drug_w_paths:



# A tibble: 3 × 2
  ModelID name               
  <chr>   <chr>              
1 DIPG10  3-deazaneplanocin-a
2 DIPG10  axitinib           
3 DIPG10  cladribine         


In [10]:
plot_df <- cell_killing_df %>%
  left_join(final_drug_w_paths, by = c("ModelID", "name")) %>%
  filter(name %in% c("3-deazaneplanocin-a", "cladribine", "axitinib")) %>%
  mutate(
    one_minus_auc = 1 - auc,
    drug = factor(name,
      levels = c("3-deazaneplanocin-a", "cladribine", "axitinib"),
      labels = c("3-Deazaneplanocin-A", "Cladribine", "Axitinib")
    ),
    has_latent = !is.na(latent_score)
  )
 
message("\nFinal plot_df row counts per drug (non-NA latent_score):")
print(plot_df %>% filter(!is.na(latent_score)) %>% count(drug))



Final plot_df row counts per drug (non-NA latent_score):



# A tibble: 3 × 2
  drug                    n
  <fct>               <int>
1 3-Deazaneplanocin-A     8
2 Cladribine              8
3 Axitinib                8


In [11]:
plot_df$ModelID <- factor(plot_df$ModelID)
samples <- levels(plot_df$ModelID)
sample_colors <- setNames(
  qualitative_hcl(length(samples), palette = "Pastel 1"),
  samples
)


In [12]:
make_panel <- function(data, y_var, y_label, is_log = FALSE) {
  d <- data %>% filter(!is.na(latent_score), !is.na(.data[[y_var]]))
 
  if (nrow(d) < 2) {
    # Not enough points to fit a line -- return an empty/annotated panel
    # instead of letting lm() error out
    return(
      ggplot() +
        annotate("text", x = 0.5, y = 0.5,
                 label = paste0("Insufficient data\n(n=", nrow(d), ")"),
                 size = 3, color = "#999999") +
        theme_void()
    )
  }
 
  fit  <- lm(as.formula(paste(y_var, "~ latent_score")), data = d)
  r2   <- summary(fit)$r.squared
  
  annot <- paste0("R\u00b2 = ", round(r2, 2))
 
  p <- ggplot(d, aes(x = latent_score, y = .data[[y_var]], fill = ModelID)) +
    geom_smooth(aes(group = 1), method = "lm", se = FALSE,
                color = "#272bf3", linewidth = 1.2, linetype = "dashed") +
    geom_point(shape = 21, size = 6, stroke = 0.8, color = "black") +
    geom_text_repel(
        aes(label = ModelID),
        size = 5,
        color = "#333333",

        #  key spacing controls
        box.padding = 1,
        point.padding = 0.8,

        # stronger separation
        force = 1.5,
        force_pull = 0.3,

        # allow more movement before stopping
        max.overlaps = Inf,

        # cleaner aesthetics
        segment.color = "#BBBBBB",
        segment.size = 0.4,

        # helps spread labels more evenly
        min.segment.length = 0
      ) +
    annotate("label", x = Inf, y = Inf, label = annot,
             hjust = 1.1, vjust = 13, size = 5,
             color = "#1a1717", fill = "white",
             label.size = 0.3, label.padding = unit(0.3, "lines")) +
    scale_fill_manual(values = sample_colors) +
    labs(x = "Latent Score", y = y_label, fill = "Sample") +
    theme_minimal(base_size = 18) +
    theme(
      panel.background = element_rect(fill = "white", color = "#DDDDDD"),
      plot.background  = element_rect(fill = "#F8F9FA", color = NA),
      panel.grid.major = element_line(color = "#CCCCCC", linewidth = 0.4),
      panel.grid.minor = element_blank(),
      axis.title = element_text(size = 18, color = "#555555"),
      axis.text  = element_text(size = 15, color = "#444444"),
      legend.position = "none"
    )
 
  if (is_log) {
    p <- p + scale_y_continuous(labels = function(v) paste0("1/", round(10^(-v), 0)))
  }
 
  p
}


In [13]:
drugs_display <- c("3-Deazaneplanocin-A", "Cladribine", "Axitinib")
 
auc_panels <- map(drugs_display, \(d) {
  make_panel(filter(plot_df, drug == d), "one_minus_auc",
             "1 \u2013 AUC (higher = more sensitive)") +
    ggtitle(d) +
    theme(plot.title = element_text(face = "bold", size = 20, color = "#1A1A2E"))
})
 


In [15]:
auc_row <- wrap_plots(auc_panels, ncol = 3)

final_plot <- auc_row +
  plot_annotation(
    title = "Drug Latent Score vs. Cell-Killing Activity",
    subtitle = "Higher values = greater sensitivity",
    theme = theme(
      plot.title = element_text(face = "bold", size = 25, color = "#1A1A2E"),
      plot.subtitle = element_text(size = 18, color = "#666666"),
      plot.background = element_rect(fill = "#F8F9FA", color = NA)
    )
  )
 
ggsave("./visualize/drug_latent_vs_killing.png", final_plot,
       width = 15, height = 7, dpi = 180, bg = "#F8F9FA")
 
message("Saved: drug_latent_vs_killing.png")


`geom_smooth()` using formula = 'y ~ x'
Warning message:
“The following aesthetics were dropped during statistical transformation: fill.
ℹ This can happen when ggplot fails to infer the correct grouping structure in
  the data.
ℹ Did you forget to specify a `group` aesthetic or to convert a numerical
  variable into a factor?”
`geom_smooth()` using formula = 'y ~ x'
Warning message:
“The following aesthetics were dropped during statistical transformation: fill.
ℹ This can happen when ggplot fails to infer the correct grouping structure in
  the data.
ℹ Did you forget to specify a `group` aesthetic or to convert a numerical
  variable into a factor?”
`geom_smooth()` using formula = 'y ~ x'
Warning message:
“The following aesthetics were dropped during statistical transformation: fill.
ℹ This can happen when ggplot fails to infer the correct grouping structure in
  the data.
ℹ Did you forget to specify a `group` aesthetic or to convert a numerical
  variable into a factor?”
Saved: drug_l